# ForenSight: Semantic-Only AI-Generated Image Detector
### Complete Standalone Implementation for Kaggle GPU

This notebook implements the complete, leak-free **Semantic-Only** detector from the **ForenSight** research project:
- **Backbone**: OpenCLIP ViT-L/14 (`ViT-L-14-quickgelu`, pretrained `openai`) with frozen visual representation.
- **Classification Head**: Linear projection ($768 \to 256$) $\to$ LayerNorm $\to$ Linear classifier ($256 \to 1$).
- **Dataset**: `yangsangtai/tiny-genimage` (Single protocol: Train on SD1.5, Test on In-Domain SD1.5 + 6 unseen OOD generators: ADM, BigGAN, GLIDE, Midjourney, VQDM, Wukong).
- **Leakage Policy**: Pre-flight zero-leakage validation, validation-only threshold calibration ($\tau^*$), dual-threshold evaluation ($\tau^*$ vs fixed $0.5$).
- **Optimizer & Schedule**: AdamW (lr=1e-3, weight_decay=1e-4), batch_size=16, 5 epochs.


In [ ]:
# 1. Environment Setup & Dependencies
import sys
import subprocess

packages = ["open_clip_torch", "kagglehub", "scikit-learn", "timm", "pandas"]
for pkg in packages:
    try:
        mod = pkg.replace("-torch", "").replace("-learn", "").split("_")[0]
        __import__(mod)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os
import gc
import json
import random
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, 
    f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
)
import open_clip
import kagglehub

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# 2. Strict Determinism & Seed Governance
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Deterministic seed locked to {SEED}")


In [ ]:
# 3. Dataset Discovery & Auto-Download
KAGGLE_INPUT_DIR = Path("/kaggle/input/tiny-genimage")
DATASET_REF = "yangsangtai/tiny-genimage"

if KAGGLE_INPUT_DIR.exists():
    raw_root = KAGGLE_INPUT_DIR
    print(f"Found mounted Kaggle dataset at: {raw_root}")
else:
    print(f"Mounted dataset not found. Downloading {DATASET_REF} via kagglehub...")
    download_path = kagglehub.dataset_download(DATASET_REF)
    raw_root = Path(download_path)
    print(f"Dataset downloaded to: {raw_root}")

subdirs = [p.name for p in raw_root.iterdir() if p.is_dir() and not p.name.startswith(".")]
print(f"Discovered {len(subdirs)} generator directories: {sorted(subdirs)}")


In [ ]:
# 4. Dataset Scanning & Single Protocol Partitioning
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def scan_dataset(root: Path):
    records = []
    for gen_dir in root.iterdir():
        if not gen_dir.is_dir() or gen_dir.name.startswith("."):
            continue
        gen_name = gen_dir.name.lower()
        if "adm" in gen_name: g = "adm"
        elif "biggan" in gen_name: g = "biggan"
        elif "glide" in gen_name: g = "glide"
        elif "midjourney" in gen_name: g = "midjourney"
        elif "vqdm" in gen_name: g = "vqdm"
        elif "wukong" in gen_name: g = "wukong"
        elif any(k in gen_name for k in ["sdv1_5", "sd15", "sd_1_5", "stable_diffusion"]): g = "sd15"
        else: g = gen_name

        for img_path in gen_dir.rglob("*"):
            if img_path.suffix.lower() in IMAGE_EXTENSIONS and not img_path.name.startswith("._"):
                rel_parts = [p.lower() for p in img_path.relative_to(gen_dir).parts]
                is_real = any(p in ("nature", "real", "0_real") for p in rel_parts)
                is_fake = any(p in ("ai", "fake", "1_fake") for p in rel_parts)
                if is_real:
                    label = 0
                    sample_generator = "nature"
                elif is_fake:
                    label = 1
                    sample_generator = g
                else:
                    continue

                stem = img_path.stem
                class_id = stem.split("_")[0] if (stem.startswith("n") and len(stem) > 9) else "unknown"
                
                records.append({
                    "sample_id": f"{g}_{label}_{img_path.name}",
                    "path": str(img_path),
                    "label": label,
                    "generator": sample_generator,
                    "source_generator": g,
                    "class_id": class_id,
                })
    return records

print("Scanning dataset files...")
all_records = scan_dataset(raw_root)
print(f"Total valid images scanned: {len(all_records):,}")

# Partition according to Single Protocol:
# Train: 4000 SD1.5 (2000 fake + 2000 real)
# Val: 500 SD1.5 (250 fake + 250 real)
# In-Domain Test: 500 SD1.5 (250 fake + 250 real)
# OOD Test: 6000 across 6 unseen generators (1000 each: 500 fake + 500 real)
rng = random.Random(SEED)

sd15_fakes = [r for r in all_records if r["source_generator"] == "sd15" and r["label"] == 1]
sd15_reals = [r for r in all_records if r["source_generator"] == "sd15" and r["label"] == 0]
rng.shuffle(sd15_fakes)
rng.shuffle(sd15_reals)

train_records = sd15_fakes[:2000] + sd15_reals[:2000]
val_records = sd15_fakes[2000:2250] + sd15_reals[2000:2250]
in_domain_test_records = sd15_fakes[2250:2500] + sd15_reals[2250:2500]

for r in train_records: r["split"] = "train"
for r in val_records: r["split"] = "val"
for r in in_domain_test_records: r["split"] = "in_domain_test"

ood_generators = ["adm", "biggan", "glide", "midjourney", "vqdm", "wukong"]
ood_test_records = []

for g in ood_generators:
    g_fakes = [r for r in all_records if r["source_generator"] == g and r["label"] == 1]
    g_reals = [r for r in all_records if r["source_generator"] == g and r["label"] == 0]
    rng.shuffle(g_fakes)
    rng.shuffle(g_reals)
    cohort = g_fakes[:500] + g_reals[:500]
    for r in cohort:
        r["split"] = "cross_generator_ood"
        r["eval_generator"] = g
    ood_test_records.extend(cohort)

print(f"
Manifest Split Summary:")
print(f"  - Train:        {len(train_records):,} (2000 Fake + 2000 Real)")
print(f"  - Val:          {len(val_records):,} (250 Fake + 250 Real)")
print(f"  - In-Domain:    {len(in_domain_test_records):,} (250 Fake + 250 Real)")
print(f"  - OOD Test:     {len(ood_test_records):,} across 6 generators (500 Fake + 500 Real each)")

# Leakage audit
train_paths = {r["path"] for r in train_records}
val_paths = {r["path"] for r in val_records}
in_paths = {r["path"] for r in in_domain_test_records}
ood_paths = {r["path"] for r in ood_test_records}
assert train_paths.isdisjoint(val_paths), "Train and Val overlap!"
assert train_paths.isdisjoint(in_paths), "Train and In-domain test overlap!"
assert train_paths.isdisjoint(ood_paths), "Train and OOD test overlap!"
assert val_paths.isdisjoint(in_paths), "Val and In-domain test overlap!"
assert val_paths.isdisjoint(ood_paths), "Val and OOD test overlap!"
print("
[PASS] Zero-Leakage Audit: All partitions are strictly disjoint.")


In [ ]:
# 5. Dataset Loader & SemanticOnlyDetector Architecture
class ImageDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        with Image.open(item["path"]) as img:
            tensor = self.transform(img.convert("RGB"))
        label = torch.tensor(item["label"], dtype=torch.float32)
        return tensor, label, item

class SemanticOnlyDetector(nn.Module):
    """
    OpenCLIP ViT-L/14 Backbone with frozen visual weights.
    Linear projection: 768 -> 256 -> LayerNorm -> Linear classifier: 256 -> 1.
    """
    def __init__(self, clip_model_name: str = "ViT-L-14-quickgelu", pretrained: str = "openai", projection_dim: int = 256):
        super().__init__()
        clip_model, _, preprocess = open_clip.create_model_and_transforms(
            clip_model_name, pretrained=pretrained
        )
        self.visual = clip_model.visual
        self.preprocess = preprocess

        # Strictly freeze OpenCLIP visual representation
        for param in self.visual.parameters():
            param.requires_grad = False
        self.visual.eval()

        feature_dim = getattr(self.visual, "output_dim", 768)

        # Trainable projection layer and binary classification head
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim)
        )
        self.classifier = nn.Linear(projection_dim, 1)

    def forward(self, x):
        with torch.no_grad():
            features = self.visual(x)
        projected = self.projection(features)
        logits = self.classifier(projected)
        return logits.squeeze(-1)

print("Initializing OpenCLIP ViT-L-14-quickgelu backbone...")
model = SemanticOnlyDetector().to(device)
preprocess = model.preprocess

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters:    {frozen_params:,}")


In [ ]:
# 6. Evaluation Utilities & Zero-Leakage Threshold Calibration
def predict_dataset(model, loader):
    model.eval()
    scores, targets, metadata = [], [], []
    with torch.no_grad():
        for images, labels, items in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy()
            scores.extend(probs)
            targets.extend(labels.numpy())
            for i in range(len(labels)):
                metadata.append({
                    "sample_id": items["sample_id"][i],
                    "label": int(labels[i].item()),
                    "score": float(probs[i]),
                    "split": items["split"][i],
                    "generator": items["generator"][i],
                    "eval_generator": items.get("eval_generator", [items["generator"][i]])[i],
                })
    return np.array(scores), np.array(targets), metadata

def compute_metrics(scores, targets, threshold=0.5):
    auroc = roc_auc_score(targets, scores)
    p, r, _ = precision_recall_curve(targets, scores)
    pr_auc = auc(r, p)
    preds = (scores >= threshold).astype(int)
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, zero_division=0)
    prec = precision_score(targets, preds, zero_division=0)
    rec = recall_score(targets, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(targets, preds, labels=[0, 1]).ravel()
    return {
        "auroc": float(auroc),
        "pr_auc": float(pr_auc),
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall": float(rec),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)
    }

def calibrate_tau_star(scores, targets):
    best_tau = 0.5
    best_f1 = -1.0
    for tau in np.linspace(0.01, 0.99, 100):
        preds = (scores >= tau).astype(int)
        score = f1_score(targets, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_tau = tau
    return float(best_tau)


In [ ]:
# 7. Training Engine (5 Epochs, AdamW lr=1e-3)
BATCH_SIZE = 16
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
EPOCHS = 5

train_loader = DataLoader(ImageDataset(train_records, preprocess), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(ImageDataset(val_records, preprocess), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=1e-4
)

OUTPUT_DIR = Path("/kaggle/working/results_semantic_seed42")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_checkpoint_path = OUTPUT_DIR / "checkpoint.pt"

best_val_auroc = 0.0
train_history = []

print(f"Starting training for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        
    train_loss = total_loss / len(train_records)
    
    # Validation evaluation
    val_scores, val_targets, _ = predict_dataset(model, val_loader)
    val_metrics = compute_metrics(val_scores, val_targets)
    val_auroc = val_metrics["auroc"]
    
    train_history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_auroc": round(val_auroc, 4),
    })
    
    print(f"Epoch {epoch:03d}/{EPOCHS:03d} | Train Loss: {train_loss:.4f} | Val AUROC: {val_auroc:.4f}")
    
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_auroc": best_val_auroc,
            "seed": SEED,
        }, best_checkpoint_path)
        print(f"  -> [BEST CHECKPOINT] Saved epoch {epoch} (Val AUROC: {best_val_auroc:.4f})")

print(f"
Training complete! Peak Validation AUROC: {best_val_auroc:.4f}")


In [ ]:
# 8. Leak-Free Evaluation & Reporting (In-Domain + 6 OOD Generators)
# Load peak checkpoint
print(f"Loading best weights from {best_checkpoint_path}...")
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Calibrate tau* on validation set
val_scores, val_targets, _ = predict_dataset(model, val_loader)
tau_star = calibrate_tau_star(val_scores, val_targets)
print(f"Validation-Calibrated Decision Threshold (tau*): {tau_star:.4f}
")

# Predict In-Domain & OOD test sets
in_loader = DataLoader(ImageDataset(in_domain_test_records, preprocess), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
ood_loader = DataLoader(ImageDataset(ood_test_records, preprocess), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_scores, in_targets, _ = predict_dataset(model, in_loader)
ood_scores, ood_targets, ood_meta = predict_dataset(model, ood_loader)

# Overall metrics
in_metrics_tau = compute_metrics(in_scores, in_targets, threshold=tau_star)
in_metrics_05 = compute_metrics(in_scores, in_targets, threshold=0.5)

ood_metrics_tau = compute_metrics(ood_scores, ood_targets, threshold=tau_star)
ood_metrics_05 = compute_metrics(ood_scores, ood_targets, threshold=0.5)

# Per-generator OOD metrics
table_rows = []
for g in ood_generators:
    g_idx = [i for i, m in enumerate(ood_meta) if m["eval_generator"] == g]
    g_scores = ood_scores[g_idx]
    g_targets = ood_targets[g_idx]
    m_tau = compute_metrics(g_scores, g_targets, threshold=tau_star)
    m_05 = compute_metrics(g_scores, g_targets, threshold=0.5)
    table_rows.append({
        "Generator": g.upper(),
        "AUROC": f"{m_tau['auroc']:.4f}",
        "PR-AUC": f"{m_tau['pr_auc']:.4f}",
        "Acc (tau*)": f"{m_tau['accuracy']*100:.2f}%",
        "F1 (tau*)": f"{m_tau['f1']:.4f}",
        "Acc (0.5)": f"{m_05['accuracy']*100:.2f}%",
        "F1 (0.5)": f"{m_05['f1']:.4f}",
    })

print("="*65)
print(f"          FORENSIGHT EVALUATION REPORT (Semantic-Only, Seed {SEED})")
print("="*65)
print(f"Validation AUROC:              {best_val_auroc:.4f}")
print(f"In-Domain (SD 1.5) AUROC:      {in_metrics_tau['auroc']:.4f}")
print(f"Cross-Generator OOD AUROC:     {ood_metrics_tau['auroc']:.4f}")
print(f"Calibrated Threshold (tau*):    {tau_star:.4f}")
print(f"OOD Accuracy @ tau*:           {ood_metrics_tau['accuracy']*100:.2f}% (F1: {ood_metrics_tau['f1']:.4f})")
print(f"OOD Accuracy @ tau=0.5:        {ood_metrics_05['accuracy']*100:.2f}% (F1: {ood_metrics_05['f1']:.4f})")
print("="*65)

df_per_gen = pd.DataFrame(table_rows)
display(df_per_gen)


In [ ]:
# 9. Artifact Export
summary_dict = {
    "model": "SemanticOnlyDetector",
    "backbone": "OpenCLIP ViT-L/14-quickgelu (openai)",
    "seed": SEED,
    "epochs": EPOCHS,
    "val_auroc": best_val_auroc,
    "in_domain_auroc": in_metrics_tau["auroc"],
    "ood_auroc": ood_metrics_tau["auroc"],
    "calibrated_threshold": tau_star,
    "in_domain_tau": in_metrics_tau,
    "in_domain_05": in_metrics_05,
    "ood_tau": ood_metrics_tau,
    "ood_05": ood_metrics_05,
    "per_generator": table_rows,
    "train_history": train_history,
}

with open(OUTPUT_DIR / "evaluation.json", "w") as f:
    json.dump(summary_dict, f, indent=2)

pd.DataFrame(ood_meta).to_csv(OUTPUT_DIR / "ood_predictions.csv", index=False)

print(f"
Artifacts successfully written to {OUTPUT_DIR}:")
for p in OUTPUT_DIR.iterdir():
    print(f"  - {p.name:25s} ({p.stat().st_size:,} bytes)")
